In [0]:
spark.sql("USE CATALOG wsshubhamcontest")
VOLUME_PATH = "/Volumes/wsshubhamcontest/supply_chain/raw_data"

In [0]:
from pyspark.sql.functions import col, trim, upper, when, lit
from pyspark.sql.types import StringType

sap_tables = [
    "MARA", "MARC", "MARD", "MAKT", "EKKO", "EKPO", "EKET", "EKBE",
    "LFA1", "LFB1", "LFM1", "MBEW", "MBEWH", "MARDH", "MARCH",
    "MCHB", "MCHBH", "MCH1", "MSLB", "MSLBH", "QALS", "QAVE",
    "AUSP", "AUSP_BATCH", "PLKO", "T001", "T001K", "T001L", "T001W",
    "T023T", "T024", "T024E", "T077K", "T134T", "TCURF", "TQ30T", "TQ31T"
]

for table in sap_tables:
    df = spark.table(f"wsshubhamcontest.bronze.{table.lower()}")
    
    # Drop duplicates
    df = df.dropDuplicates()
    
    # Trim all string columns
    for c in df.columns:
        if df.schema[c].dataType == StringType():
            df = df.withColumn(c, trim(col(c)))
    
    # Drop rows where all values are null
    df = df.dropna(how='all')
    
    df.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable(f"wsshubhamcontest.silver.{table.lower()}")
    
    print(f"✅ Silver: {table} — {df.count()} rows")

✅ Silver: MARA — 10000 rows
✅ Silver: MARC — 10000 rows
✅ Silver: MARD — 10000 rows
✅ Silver: MAKT — 10000 rows
✅ Silver: EKKO — 10000 rows
✅ Silver: EKPO — 10000 rows
✅ Silver: EKET — 10000 rows
✅ Silver: EKBE — 10000 rows
✅ Silver: LFA1 — 10000 rows
✅ Silver: LFB1 — 10000 rows
✅ Silver: LFM1 — 10000 rows
✅ Silver: MBEW — 10000 rows
✅ Silver: MBEWH — 10000 rows
✅ Silver: MARDH — 10000 rows
✅ Silver: MARCH — 10000 rows
✅ Silver: MCHB — 10000 rows
✅ Silver: MCHBH — 10000 rows
✅ Silver: MCH1 — 10000 rows
✅ Silver: MSLB — 10000 rows
✅ Silver: MSLBH — 10000 rows
✅ Silver: QALS — 0 rows
✅ Silver: QAVE — 10000 rows
✅ Silver: AUSP — 9987 rows
✅ Silver: AUSP_BATCH — 280196 rows
✅ Silver: PLKO — 10000 rows
✅ Silver: T001 — 10000 rows
✅ Silver: T001K — 10000 rows
✅ Silver: T001L — 10000 rows
✅ Silver: T001W — 10000 rows
✅ Silver: T023T — 10000 rows
✅ Silver: T024 — 10000 rows
✅ Silver: T024E — 10000 rows
✅ Silver: T077K — 5 rows
✅ Silver: T134T — 24 rows
✅ Silver: TCURF — 10000 rows
✅ Silver: TQ

In [0]:
ibp_tables = [
    "demand_actual", "demand_forcast", "master_customer",
    "master_customer_product", "master_location", "master_location_product"
]

for table in ibp_tables:
    df = spark.table(f"wsshubhamcontest.bronze.{table.lower()}")
    
    # Drop duplicates
    df = df.dropDuplicates()
    
    # Trim all string columns
    for c in df.columns:
        if df.schema[c].dataType == StringType():
            df = df.withColumn(c, trim(col(c)))
    
    # Drop rows where all values are null
    df = df.dropna(how='all')
    
    df.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable(f"wsshubhamcontest.silver.{table.lower()}")
    
    print(f"✅ Silver: {table} — {df.count()} rows")

✅ Silver: demand_actual — 10000 rows
✅ Silver: demand_forcast — 10000 rows
✅ Silver: master_customer — 10000 rows
✅ Silver: master_customer_product — 10000 rows
✅ Silver: master_location — 10000 rows
✅ Silver: master_location_product — 10000 rows


In [0]:
display(spark.sql("SHOW TABLES IN wsshubhamcontest.silver"))

database,tableName,isTemporary
silver,ausp,false
silver,ausp_batch,false
silver,demand_actual,false
silver,demand_forcast,false
silver,ekbe,false
silver,eket,false
silver,ekko,false
silver,ekpo,false
silver,lfa1,false
silver,lfb1,false


In [0]:
# See any silver table
display(spark.table("wsshubhamcontest.silver.mara"))

AENAM,BISMT,BREIT,BRGEW,BSTME,DATAB,EAN11,ERNAM,ERSDA,GEWEI,HERKL,HOEHE,KZKFG,LAEDA,LAENG,LIQDT,LVORM,MANDT,MATKL,MATNR,MBRSH,MEINS,MFRNR,MSTAE,MSTDE,MTART,NTGEW,PRDHA,PSTAT,QMPUR,SPART,VOLEH,VOLUM,VPSTA,XCHPF
lindawilliam,109186386317336007,80.519,278.843,KG,20250719,7362243483201,amber95,20230101,KG,IN,66.096,null,20220314,88.66,20220528,null,100,812306831,667826226751862899,A,L,1000000026,3,20220517,HAWA,195.197,803275785939849993,K,X,3,M3,0.472,K,X
gerald01,773206488772444239,53.877,260.481,M,20220227,9818246289690,alexbutler,20251217,KG,CN,45.846,null,20210424,35.514,20211004,null,100,213851780,600530520645402489,C,M,1000000398,2,20220717,FERT,217.888,683677636534744718,K,X,3,M3,0.088,K,null
sandra53,366194072961420330,81.661,455.826,L,20210201,1456778891728,jonessarah,20220612,KG,IN,75.227,X,20240530,38.975,20241127,null,100,189209837,448169766557728491,A,M,1000000116,1,20231216,HAWA,371.445,321856618954465210,K,X,3,M3,0.239,K,null
kimpaul,658261737746225638,12.237,199.594,EA,20220508,4601273612603,catherine40,20210415,KG,IN,5.992,X,20210303,94.365,20250113,null,100,375496420,180461395351568868,A,M,1000000095,2,20240306,HAWA,148.594,902666149852579135,K,null,1,M3,0.007,K,X
hernandezwil,404704594168742873,2.96,429.012,KG,20221113,7252162606834,jameshansen,20250102,KG,CN,64.095,X,20230419,7.603,20221218,null,100,626841745,572597988670400239,C,L,1000000154,2,20220107,FERT,331.001,598881838113950566,K,X,2,M3,0.001,K,null
xrivera,267015598626230429,96.66,121.12,EA,20211117,1569829952362,reginaromero,20230614,KG,IN,83.564,X,20251005,89.826,20251016,null,100,154933113,291745941402491402,A,L,1000000312,3,20231209,HALB,86.574,531847252888099742,K,X,1,M3,0.726,K,null
gonzalezerin,944054421527967265,1.537,65.861,KG,20220609,9378885297150,marisa12,20240122,KG,DE,10.198,X,20210328,26.106,20240613,null,100,134933093,813231554812956243,M,KG,1000000011,1,20230817,HALB,55.641,971815661282168172,K,X,3,M3,0.0,K,null
rhonda15,826852926681183695,66.918,320.446,M,20211210,5337647157117,smonroe,20210504,KG,DE,43.406,null,20241101,63.654,20230722,null,100,421979938,825251757703815433,M,KG,1000000426,2,20251104,ROH,250.908,425153803369833741,K,X,3,M3,0.185,K,X
ashleywilson,433161023854768533,17.653,318.214,L,20250827,6485608157195,whill,20220209,KG,IN,53.581,null,20211112,64.606,20250502,null,100,198746991,319968625714157169,A,KG,1000000286,1,20210619,HAWA,266.375,901694757482615615,K,null,3,M3,0.061,K,null
erinjones,809184659181173890,87.19,400.571,EA,20221104,1807664552093,cynthia41,20220416,KG,CN,18.884,X,20230711,44.279,20230602,null,100,606834982,606199262973386003,A,M,1000000040,3,20230123,HALB,359.782,555098603500720471,K,null,1,M3,0.073,K,X


In [0]:
# Check what language code is in makt
display(spark.sql("SELECT DISTINCT SPRAS FROM wsshubhamcontest.silver.makt LIMIT 5"))

SPRAS
F
D
E
S


In [0]:
%sql
SELECT COUNT(*), 
       COUNT(WERKS) as plant_count,
       COUNT(MATNR) as material_count
FROM wsshubhamcontest.silver.ekpo
LIMIT 1;


COUNT(*),plant_count,material_count
10000,10000,10000


In [0]:
%sql
SELECT EBELN, WERKS, MATNR 
FROM wsshubhamcontest.silver.ekpo 
LIMIT 3;

EBELN,WERKS,MATNR
4500000002,1200,100010
4500000002,3100,100012
4500000002,2100,100014


In [0]:
%sql
SELECT EBELN 
FROM wsshubhamcontest.silver.ekko 
LIMIT 3;

EBELN
4595004919
4594876795
4578589106


In [0]:
%sql
SELECT COUNT(*) as matching_records
FROM wsshubhamcontest.silver.ekko k
INNER JOIN wsshubhamcontest.silver.ekpo p 
ON k.EBELN = p.EBELN;

matching_records
0


In [0]:
%sql DESCRIBE TABLE wsshubhamcontest.silver.ekpo


col_name,data_type,comment
MANDT,int,null
EBELN,bigint,null
EBELP,int,null
AEDAT,int,null
TXZ01,string,null
MATNR,int,null
BUKRS,int,null
WERKS,int,null
LGORT,int,null
MATKL,string,null


In [0]:
from pyspark.sql.functions import trim
from pyspark.sql.types import StringType

df = spark.table("wsshubhamcontest.bronze.qals")
df = df.dropDuplicates()
for c in df.columns:
    if df.schema[c].dataType == StringType():
        df = df.withColumn(c, trim(df[c]))
df = df.dropna(how='all')

df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("wsshubhamcontest.silver.qals")

print(f"✅ Silver QALS: {df.count()} rows")

✅ Silver QALS: 10000 rows
